# 第 13 课｜Embedding、Token ID 与交叉熵

**把文字、向量、预测、损失和参数更新连成一条完整的计算链。**

第 12 课训练了一个分类 MLP。本课把“类别”换成“词表里的下一个 token”，完成一个可以训练、生成并保存的微型字符语言模型。

本课只用 PyTorch 和标准库，CPU 即可；不需要下载模型或语料，也不需要上一课的文件。
**Notebook 和同名 `.py` 都可以独立使用。** Notebook 有完整解释、可执行示例、独立练习和答案；脚本偏重完整流程、输入检查和自动验收。

| 学习目标 | 你最后应当能做什么 |
|---|---|
| Token ID 与 Embedding | 解释编号为何不是语义数值；实现按编号查可训练的向量表 |
| 预测目标 | 给一串 token 正确构造输入与“下一个 token”标签 |
| 多分类交叉熵 | 区分 logits、概率、类别标签；正确处理 `[B,T,V]` |
| 训练与生成 | 训练模型，然后按最后一个位置的输出逐步生成 |
| 模型边界 | 证明本课模型只看当前字符；不把拟合四句话当作语言理解 |

建议分三次学习：第一段到交叉熵，第二段完成训练与生成，第三段独立练习。
**重点不是记住类名，而是能解释每个张量的形状、标签的含义以及哪些参数发生了变化。**

## 0. 环境与使用方法

沿用第 11、12 课的环境即可。本课的核心代码已在 **PyTorch 2.10.0+cpu** 验证；这不是要求你重装到这一版本。

尚未安装 PyTorch 时，在终端执行 `python -m pip install torch`。Jupyter 已经能用就无需重复安装。
如果 Notebook 提示找不到 torch，可在新单元执行 `%pip install torch`，安装后重启内核。
CUDA 安装选择见文末官方入口；今天先用 CPU 完成所有计算。

从上到下运行，或选择“重启内核并运行全部”。运行全部只验证示例与参考答案，**不会自动完成你的练习**。
本课会在当前工作目录创建 `lesson_13_outputs/notebook/`，并覆盖其中本课生成的同名文件；不读取或修改旧课数据。

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path
import tempfile

import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.set_printoptions(precision=4, sci_mode=False)
torch.manual_seed(13)
device = torch.device("cpu")

print("PyTorch：", torch.__version__)
print("本课设备：", device)
print("CUDA 可用：", torch.cuda.is_available())

## 1. 建立词表：编号是索引，不是大小或语义

使用四句手工构造的短文本：

```text
我爱学习。
我爱模型。
你爱学习。
你爱模型。
```

这里**每个普通中文字符或标点对应一个 token**。这是教学编码器，不是 BPE 或某个主流模型的 tokenizer。

三个特殊 token 的编号固定：`<unk> → 0`、`<bos> → 1`、`<eos> → 2`，其余字符按首次出现顺序编号。
`<bos>` 表示句子开始，`<eos>` 表示句子结束，`<unk>` 表示词表里没有的输入字符。
这些字符串没有 Python 内置的魔法，作用完全取决于我们的数据构造与生成代码。

复习列表推导式：`[expression for item in items]` 与“建立空列表、循环、append”的思路相同；
`mapping.get(key, default)` 表示查到键时取对应值，否则取默认值。

In [2]:
SPECIAL_TOKENS = ["<unk>", "<bos>", "<eos>"]
TRAIN_TEXTS = ["我爱学习。", "我爱模型。", "你爱学习。", "你爱模型。"]

def build_vocab(texts: list[str]) -> tuple[dict[str, int], list[str]]:
    """保留字符首次出现的顺序；不要用无序集合决定编号。"""
    if not isinstance(texts, list) or not texts:
        raise ValueError("texts 必须是非空字符串列表")
    id_to_token = list(SPECIAL_TOKENS)
    for text in texts:
        if not isinstance(text, str):
            raise ValueError("texts 中的每一项必须是字符串")
        for char in text:
            if char not in id_to_token:
                id_to_token.append(char)
    token_to_id = {token: index for index, token in enumerate(id_to_token)}
    return token_to_id, id_to_token


def encode(text: str, token_to_id: dict[str, int]) -> list[int]:
    """字符级编码；不在这里添加 BOS/EOS，避免调用方重复添加。"""
    if not isinstance(text, str):
        raise ValueError("text 必须是字符串")
    if "<unk>" not in token_to_id:
        raise ValueError("词表必须含有 <unk>")
    return [token_to_id.get(char, token_to_id["<unk>"]) for char in text]


def decode(token_ids: list[int], id_to_token: list[str]) -> str:
    """保留特殊 token 的可见文字，方便检查对齐和未知字符。"""
    if not isinstance(token_ids, list):
        raise ValueError("token_ids 必须是整数列表")
    pieces = []
    for index in token_ids:
        if type(index) is not int or not 0 <= index < len(id_to_token):
            raise ValueError("编号必须是词表范围内的整数，不能是布尔值")
        pieces.append(id_to_token[index])
    return "".join(pieces)

token_to_id, id_to_token = build_vocab(TRAIN_TEXTS)
vocab_size = len(id_to_token)
for index, token in enumerate(id_to_token):
    print(index, repr(token))
print("词表大小 V =", vocab_size)
print("未知字编码：", encode("火", token_to_id))

0 '<unk>'
1 '<bos>'
2 '<eos>'
3 '我'
4 '爱'
5 '学'
6 '习'
7 '。'
8 '模'
9 '型'
10 '你'
词表大小 V = 11
未知字编码： [0]


### BOS/EOS 是作为一个编号插入的，不是逐字符编码尖括号

`encode("<bos>", token_to_id)` 会逐字符处理 `<`、`b`、`o`、`s`、`>`，不是我们要的控制编号。
正确做法是直接插入 `token_to_id["<bos>"]`。本课 `encode()` 本身不添加边界，由数据构造函数统一添加。

例如，“我爱学习。”得到完整序列：

| token | `<bos>` | 我 | 爱 | 学 | 习 | 。 | `<eos>` |
|---|---:|---:|---:|---:|---:|---:|---:|
| ID | 1 | 3 | 4 | 5 | 6 | 7 | 2 |

编号只决定查哪一行表。把“我”的 ID 乘以 2，并不能得到语义上“两倍的我”。

In [ ]:
text = "我爱学习。"
full_ids = [token_to_id["<bos>"]] + encode(text, token_to_id) + [token_to_id["<eos>"]]
print("完整编号：", full_ids)
print("边界可见：", decode(full_ids, id_to_token))

## 2. 输入与标签：错开一位，但只错开一次

训练目标是让每个位置预测下一个 token。

| 输入 x | `<bos>` | 我 | 爱 | 学 | 习 | 。 |
|---|---|---|---|---|---|---|
| 目标 y | 我 | 爱 | 学 | 习 | 。 | `<eos>` |

对应代码：

```python
x = full_ids[:, :-1]  # 不含最后一个位置
y = full_ids[:, 1:]   # 不含第一个位置
```

四句都是 5 个字符，加 BOS/EOS 后每句 7 个位置，得到 `x.shape = y.shape = [4,6]`。
**x 与 y 已经对齐，后面计算损失时不能再裁掉一位。** 错误地再移位，会训练“下下个 token”。

本课为了只关注目标对齐，要求一个批次里的文本等长；输入变长文本会明确报错，不会悄悄截断。
空字符串单独构造时仍可表示 `BOS → EOS`。padding、有效 token 掩码留到下一课。

In [ ]:
def make_lm_batch(
    texts: list[str], token_to_id: dict[str, int], device: str | torch.device = "cpu"
) -> tuple[torch.Tensor, torch.Tensor]:
    """每句先加边界，再左移一次标签；本课要求等长，不做 padding。

    例如完整序列 [BOS, 我, 爱, EOS]：
    x = [BOS, 我, 爱]；y = [我, 爱, EOS]。
    本函数已经完成移位，损失函数里不能再移位。
    """
    if not isinstance(texts, list) or not texts:
        raise ValueError("texts 必须是非空字符串列表")
    if any(token not in token_to_id for token in SPECIAL_TOKENS):
        raise ValueError("词表缺少 <unk>、<bos> 或 <eos>")
    sequences = []
    for text in texts:
        ids = [token_to_id["<bos>"]] + encode(text, token_to_id)
        ids.append(token_to_id["<eos>"])
        sequences.append(ids)
    if len({len(ids) for ids in sequences}) != 1:
        raise ValueError("本课只批量处理等长文本；变长文本与 padding 留到下一课")
    full_ids = torch.tensor(sequences, dtype=torch.long, device=device)
    return full_ids[:, :-1].contiguous(), full_ids[:, 1:].contiguous()

x, y = make_lm_batch(TRAIN_TEXTS, token_to_id, device)
print("x：", x)
print("y：", y)
print("形状：", tuple(x.shape), tuple(y.shape), "类型：", x.dtype)
print("第一句每个位置的监督：")
for source, target in zip(x[0].tolist(), y[0].tolist()):
    print(f"{id_to_token[source]:>5} -> {id_to_token[target]}")

## 3. `nn.Embedding`：把编号变成可学习向量

`nn.Embedding(V,D)` 内部保存一张形状为 `[V,D]` 的浮点参数表。
输入编号的形状是 `[B,T]`，输出形状就是 `[B,T,D]`；取值相当于 `weight[token_ids]`。[1]

这里的 `D` 由我们选择。每个维度没有预先规定的“情绪值”“逻辑值”等人类含义；参数在训练目标的约束下变化。
本课只有几句话，更不能把学到的向量当作高质量语义向量。

先手工填一张表，让你精确地看到查了哪一行，而不是面对随机小数猜含义。

In [ ]:
table = nn.Embedding(vocab_size, 4)
with torch.no_grad():
    table.weight.copy_(torch.arange(vocab_size * 4, dtype=torch.float32).reshape(vocab_size, 4) / 10)

lookup_ids = torch.tensor([[token_to_id["我"], token_to_id["爱"], token_to_id["我"]]], dtype=torch.long)
lookup_vectors = table(lookup_ids)
print("编号：", lookup_ids)
print("表的形状：", tuple(table.weight.shape))
print("输出形状：", tuple(lookup_vectors.shape))
print("查表结果：", lookup_vectors)
print("等于直接取行：", torch.equal(lookup_vectors, table.weight[lookup_ids]))

### 与 one-hot 的关系：理解等价，但不必真的构造大矩阵

one-hot 向量只有一个位置为 1，其余位置为 0。对于一个编号，`one_hot(id) @ E` 会选中表 `E` 的相应行。

因此对于这个普通 Embedding：

$$\mathrm{Embedding}(ids)=\mathrm{one\_hot}(ids)E.$$

这是对查表的线性代数解释。实际使用 `nn.Embedding` 时直接传整数 ID 即可，不需要先构造 one-hot。

In [ ]:
one_hot_ids = F.one_hot(lookup_ids, num_classes=vocab_size).to(torch.float32)
via_multiplication = one_hot_ids @ table.weight
print("one-hot 形状：", tuple(one_hot_ids.shape))
print("与查表相同：", torch.equal(via_multiplication, lookup_vectors))

## 4. Embedding 怎样学习？编号不变，表里的浮点参数改变

输入 ID 是离散索引。我们不对这些整数 ID 求导，而对 Embedding 的浮点权重求导。[1]

下面单独对三次查表结果求和再反向传播：同一行查到两次，该行在这次反向传播中的梯度就累加两次。
没有被查到的行，在**这个独立、无共享权重的查表实验中**梯度为 0。

不要把它扩大解释为“任何训练里没查到的行都不会更新”：权重共享、优化器历史和权重衰减等会影响真实更新。

In [ ]:
table.zero_grad(set_to_none=True)
lookup_vectors = table(lookup_ids)
lookup_vectors.sum().backward()
print("“我”查了两次，该行梯度：", table.weight.grad[token_to_id["我"]])
print("“爱”查了一次，该行梯度：", table.weight.grad[token_to_id["爱"]])
print("<unk> 没有被查，该行梯度：", table.weight.grad[token_to_id["<unk>"]])
print("整数 ID 需要梯度吗？", lookup_ids.requires_grad)

## 5. 从向量到词表分数：最小语言模型

本课模型只有两层：

```text
Token IDs     [B,T]       整数编号
    ↓ Embedding(V,D)
隐藏向量      [B,T,D]     可学习浮点向量
    ↓ Linear(D,V)
词表 logits   [B,T,V]     每个位置，对 V 个候选各打一个分数
```

`nn.Linear` 对最后一维变换，所有位置使用同一组参数。[6]
本课用 `D=16`、`V=11`，参数数量是：

$$VD+(VD+V)=11\times16+11\times16+11=363.$$

**这不是 Transformer。** 没有注意力，没有位置编码，位置之间也没有信息交换。
每个位置只根据当前 token 预测下一个 token，即 $p_\theta(x_{t+1}\mid x_t)$。
输入包含整句，不等于网络结构真的利用了整句上下文。

下面 `forward()` 中的核心只有最后两行。前面的 `if` 是便于排错的教学输入检查。

In [ ]:
class TinyNextTokenLM(nn.Module):
    """Embedding + 输出投影：只根据当前 token 预测下一个 token。

    [B,T] -> [B,T,D] -> [B,T,V]。
    位置之间没有信息混合，因此长度 T 不等于模型使用了 T 个 token 的上下文。
    """

    def __init__(self, vocab_size: int, embedding_dim: int = 16) -> None:
        super().__init__()
        if type(vocab_size) is not int or vocab_size < 1:
            raise ValueError("vocab_size 必须是正整数")
        if type(embedding_dim) is not int or embedding_dim < 1:
            raise ValueError("embedding_dim 必须是正整数")
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lm_head = nn.Linear(embedding_dim, vocab_size)

    def forward(self, token_ids: torch.Tensor) -> torch.Tensor:
        # 教学版显式检查；这些检查不是高吞吐推理服务的实现方式。
        if not isinstance(token_ids, torch.Tensor) or token_ids.ndim != 2:
            raise ValueError("输入必须是形状 [B,T] 的张量")
        if token_ids.dtype != torch.long or token_ids.numel() == 0:
            raise ValueError("输入必须是非空 torch.long 编号张量")
        if token_ids.min().item() < 0 or token_ids.max().item() >= self.vocab_size:
            raise ValueError("输入编号超出词表范围")
        hidden = self.embedding(token_ids)
        return self.lm_head(hidden)

torch.manual_seed(13)
model = TinyNextTokenLM(vocab_size, embedding_dim=16).to(device)
logits = model(x)
print("Embedding 输出：", tuple(model.embedding(x).shape))
print("logits：", tuple(logits.shape))
print("参数数量：", sum(p.numel() for p in model.parameters()))
for name, parameter in model.named_parameters():
    print(name, tuple(parameter.shape))

## 6. 从二分类走向词表多分类

第 12 课用一个分数表示二分类；现在每个位置有 `V` 个候选，需要输出 `V` 个 logits。

设一个位置的分数为 $z_0,\ldots,z_{V-1}$，正确编号为 $y$：

$$p_i=\frac{e^{z_i}}{\sum_j e^{z_j}},\qquad
L=-\log p_y=\log\sum_j e^{z_j}-z_y.$$

正确类概率为 0.8 时，损失约为 0.223；为 0.1 时，损失约为 2.303。
因此交叉熵不仅看最大分数的类别对不对，还看给正确类别多少概率。

**`F.cross_entropy` / `nn.CrossEntropyLoss` 接收未经 softmax 的 logits。**
使用类别编号标签时，标签为 `torch.long`，无需 one-hot。[2]
这是本课选用的接口；PyTorch 也支持概率分布标签，不在本课展开。

In [ ]:
example_logits = torch.tensor([[2., 0., -1.]])
example_target = torch.tensor([0], dtype=torch.long)
probabilities = torch.softmax(example_logits, dim=-1)
log_probabilities = F.log_softmax(example_logits, dim=-1)
manual_loss = -log_probabilities[0, 0]
api_loss = F.cross_entropy(example_logits, example_target)
wrong_input_loss = F.cross_entropy(probabilities, example_target)

print("概率：", probabilities)
print("手动取正确类负 log 概率：", manual_loss.item())
print("交叉熵接口：", api_loss.item())
print("把概率误当 logits 后的不同损失：", wrong_input_loss.item())
torch.testing.assert_close(manual_loss, api_loss)

### 为什么不在训练中直接写 `-log(softmax(z)[y])`？

数学上可等价，浮点数实现却可能下溢。例如分数相差 2000 时，极小的 softmax 概率可能变成精确的 0，随后 `log(0)` 得到负无穷。

`F.log_softmax` 使用更稳定的计算形式；交叉熵也避免要求先显式形成那些极小概率。[2][3]
下面故意观察一个不稳定写法，但不会让 Notebook 中断。

In [ ]:
extreme_logits = torch.tensor([[1000., 0., -1000.]])
extreme_target = torch.tensor([2], dtype=torch.long)
naive_loss = -torch.log(torch.softmax(extreme_logits, dim=-1)[0, 2])
stable_loss = F.cross_entropy(extreme_logits, extreme_target)
print("先 softmax 再 log：", naive_loss.item())
print("稳定交叉熵：", stable_loss.item())

## 7. 一整批文本如何计算损失？

我们的 `logits` 是 `[B,T,V]`，标签是 `[B,T]`。把每个位置看作一个 V 分类问题：

```python
loss = F.cross_entropy(
    logits.reshape(B * T, V),
    targets.reshape(B * T),
)
```

`reshape(-1,V)` 中的 `-1` 让 PyTorch 根据元素总数推导 `B*T`。
在当前的无 padding、无类别权重设置下，默认得到所有 `B*T` 个位置的平均损失。

另一种合法写法是把 logits 换轴为 `[B,V,T]`。PyTorch 的多维类别输入把类别轴放在第 1 维，
所以不要直接把 `[B,T,V]` 和 `[B,T]` 原样交给接口。[2]

本课标签已在 `make_lm_batch` 中移位；这里直接使用 `y`。

In [ ]:
def sequence_cross_entropy(logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    """[B,T,V] logits 与已经对齐的 [B,T] 标签 -> 所有位置平均损失。"""
    if not isinstance(logits, torch.Tensor) or logits.ndim != 3:
        raise ValueError("logits 必须是 [B,T,V] 张量")
    if not isinstance(targets, torch.Tensor) or targets.shape != logits.shape[:2]:
        raise ValueError("targets 必须是与 logits 对齐的 [B,T] 张量")
    if not logits.is_floating_point() or targets.dtype != torch.long:
        raise ValueError("logits 要用浮点数，类别编号 targets 要用 torch.long")
    if logits.device != targets.device:
        raise ValueError("logits 和 targets 必须在同一个设备")
    if logits.numel() == 0:
        raise ValueError("不能对空批量或空序列取平均交叉熵")
    vocab_size = logits.shape[-1]
    if targets.min().item() < 0 or targets.max().item() >= vocab_size:
        raise ValueError("目标编号超出词表范围")
    if not torch.isfinite(logits).all().item():
        raise ValueError("本课的 logits 必须全部有限")
    return F.cross_entropy(logits.reshape(-1, vocab_size), targets.reshape(-1))

logits = model(x)
loss = sequence_cross_entropy(logits, y)
alternative_loss = F.cross_entropy(logits.transpose(1, 2), y)
print("展平 logits：", tuple(logits.reshape(-1, vocab_size).shape))
print("展平 labels：", tuple(y.reshape(-1).shape))
print("初始 loss：", loss.item())
print("两种布局损失一致：", torch.allclose(loss, alternative_loss))

## 8. 检查梯度：哪些参数接收到学习信号？

运行 `loss.backward()` 后，Embedding 表和输出投影参数会接收梯度。
这一步还没有更新参数；只有后续 `optimizer.step()` 才执行优化器更新。

特别观察：输入中不含 EOS，但 EOS 是输出目标，所以输出层的 EOS 类仍可得到学习信号。
本模型的 Embedding 表与输出层权重**没有共享**，不能把两层里同编号的行混为一谈。

In [ ]:
model.zero_grad(set_to_none=True)
loss = sequence_cross_entropy(model(x), y)
loss.backward()
for name, parameter in model.named_parameters():
    print(name, "梯度形状", tuple(parameter.grad.shape), "梯度范数", parameter.grad.norm().item())
print("输入是整数编号：", x.dtype, "requires_grad =", x.requires_grad)
model.zero_grad(set_to_none=True)

## 9. 训练完整模型

训练循环仍是你已经学过的：`zero_grad → forward → loss → backward → step`。
我们对四句话组成的小批次重复训练 250 次。

训练时输入使用数据中真实的前一个 token；生成时，新生成的 token 会成为下一步输入。
在一般序列生成学习中，前者通常称为 teacher forcing。本课只用当前一个字符，先把这个区别看清楚。

**本课四句话全部参与训练，没有独立验证集。** 所有 loss、perplexity、准确率都明确是训练集拟合指标。
这里 perplexity 按 `exp(平均交叉熵)` 计算，只有相同分词和计分规则下才方便直接比较。

下一课再引入训练/验证划分、批量读取和有效 token 计分；此处不制造“语言能力很强”的结论。

In [ ]:
def evaluate_model(model: TinyNextTokenLM, x: torch.Tensor, y: torch.Tensor) -> dict:
    """不更新参数；恢复调用前的训练/评估模式。"""
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            logits = model(x)
            loss = sequence_cross_entropy(logits, y).item()
            accuracy = (logits.argmax(dim=-1) == y).float().mean().item()
    finally:
        model.train(was_training)
    return {"loss": loss, "perplexity": math.exp(loss), "token_accuracy": accuracy}


def train_model(
    model: TinyNextTokenLM, x: torch.Tensor, y: torch.Tensor,
    *, steps: int = 250, lr: float = 0.05,
) -> dict:
    """全量小批次训练；这里只用于验证教学链路，不做泛化评测。"""
    if type(steps) is not int or steps < 1:
        raise ValueError("steps 必须是正整数")
    if isinstance(lr, bool) or not isinstance(lr, (float, int)) or not math.isfinite(lr) or lr <= 0:
        raise ValueError("lr 必须是正的有限数")
    before = evaluate_model(model, x, y)
    history = [{"step": 0, **before}]
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for step in range(1, steps + 1):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        logits = model(x)
        loss = sequence_cross_entropy(logits, y)
        loss.backward()
        optimizer.step()
        if step % 50 == 0 or step == steps:
            history.append({"step": step, **evaluate_model(model, x, y)})
    return {"before": before, "after": history[-1], "history": history}

In [ ]:
# 重新初始化，确保独立运行脚本与本实验使用同一随机起点。
torch.manual_seed(13)
model = TinyNextTokenLM(vocab_size, embedding_dim=16).to(device)
embedding_before = model.embedding.weight.detach().clone()
training_report = train_model(model, x, y, steps=250, lr=0.05)

for record in training_report["history"]:
    print(f"step={record['step']:3d}  loss={record['loss']:.6f}  token准确率={record['token_accuracy']:.2%}")
print("Embedding 参数发生变化：", not torch.equal(embedding_before, model.embedding.weight.detach()))

## 10. 为什么 loss 不等于 0，准确率也不是 100%？

先看输入到目标的统计，而不是马上怀疑代码。

每句的 6 个预测位置里：

* `BOS` 后，“我”和“你”各占一半；
* “爱”后，“学”和“模”各占一半；
* 其余 4 个位置在这份数据上都有确定后续。

只看当前 token 的模型对相同输入必须给出相同分布，因此两个不确定位置最合理的概率各约为 0.5。
这份数据上任意一字符条件模型的平均交叉熵下界为：

$$L_{\min}=\frac{2\log 2+4\times0}{6}=\frac{\log 2}{3}\approx0.231049.$$

单一确定类别预测的训练准确率上界是：

$$\frac{2\times0.5+4\times1}{6}=\frac56\approx83.33\%.$$

有限 logits 的 softmax 通常只能接近这个损失下界。程序会直接从频数计算，而不是把答案写进训练循环。

**两个限制要分开：** 本模型没有更早上下文；而本语料本身也含相同完整前缀的多种合法后续。
例如“我爱”后两种句子都出现了，不能声称换成 Transformer 就应把这些位置全部唯一预测正确。

In [ ]:
def bigram_frequency_baseline(x: torch.Tensor, y: torch.Tensor, vocab_size: int) -> dict:
    """统计相同输入 ID 的目标频率，推导本数据上的一字符条件模型下界。

    这是使用同一份训练数据计算的诊断基线，不是独立测试集。
    未出现的输入行不参与平均；有限 softmax logits 通常只能接近此损失下界。
    """
    if x.shape != y.shape or x.numel() == 0:
        raise ValueError("x、y 必须非空且形状相同")
    counts = torch.zeros((vocab_size, vocab_size), dtype=torch.float64)
    for source, target in zip(x.detach().cpu().reshape(-1).tolist(), y.detach().cpu().reshape(-1).tolist()):
        counts[source, target] += 1
    total = x.numel()
    loss_floor = 0.0
    correct = 0
    for row in counts:
        row_total = row.sum().item()
        if row_total == 0:
            continue
        correct += int(row.max().item())
        for count in row.tolist():
            if count > 0:
                loss_floor -= count / total * math.log(count / row_total)
    return {
        "loss_floor": loss_floor,
        "accuracy_ceiling": correct / total,
        "counts": counts,
    }

baseline = bigram_frequency_baseline(x, y, vocab_size)
print("统计损失下界：", baseline["loss_floor"])
print("训练后的损失：", training_report["after"]["loss"])
print("一字符条件准确率上界：", baseline["accuracy_ceiling"])
for token in ["<bos>", "爱"]:
    row = baseline["counts"][token_to_id[token]]
    print("输入", token, "的目标频数：", {id_to_token[i]: int(c) for i, c in enumerate(row.tolist()) if c})

## 11. 生成：不断使用最后一个位置的输出

一次前向计算会给所有输入位置打分，但下一次要追加的 token 由最后一个位置的 logits 决定：

```python
last_logits = model(prefix_ids)[0, -1, :]
next_id = last_logits.argmax().item()
```

贪心选择每次取分数最大者；采样先用 `softmax(logits / temperature)` 形成分布，再调用 `torch.multinomial` 抽样。[4]
温度必须大于 0；本课把贪心作为独立选项，不用“temperature=0”触发除零。

每步把新编号接到前缀，遇到 EOS 停止；最大新增 token 数是另一道停止保护。
**生成不调用 backward，也不更新权重。** `eval()` 与 `no_grad()` 仍各自负责模式与梯度记录。

函数稍长，先抓住循环里的 `last_logits → next_id → append → EOS` 四步。
它使用局部 CPU 随机数生成器，不依赖前面单元消耗了多少随机数。复现只承诺同环境、同模型和同参数条件，跨设备/版本不保证逐位相同。

In [ ]:
def generate(
    model: TinyNextTokenLM, prompt: str, token_to_id: dict[str, int], id_to_token: list[str],
    *, max_new_tokens: int = 12, method: str = "greedy", temperature: float = 1.0,
    seed: int = 13,
) -> dict:
    """逐 token 生成；遇到 EOS 或达到预算时结束，不改变模型参数。

    为简化跨设备随机数管理，采样在 CPU 上使用局部 Generator。
    不偷偷过滤输出类别：未训练好时可能生成 <unk>/<bos>，会原样显示。
    temperature 只作用于 sample；greedy 直接选择 logits 最大的编号。
    """
    if type(max_new_tokens) is not int or max_new_tokens < 0:
        raise ValueError("max_new_tokens 必须是非负整数")
    if method not in {"greedy", "sample"}:
        raise ValueError("method 必须是 greedy 或 sample")
    if isinstance(temperature, bool) or not isinstance(temperature, (int, float)) or not math.isfinite(temperature) or temperature <= 0:
        raise ValueError("temperature 必须是正的有限数；贪心使用 method='greedy'")
    if type(seed) is not int:
        raise ValueError("seed 必须是整数")
    if len(id_to_token) != model.vocab_size or token_to_id != {t: i for i, t in enumerate(id_to_token)}:
        raise ValueError("模型与词表映射不一致")
    if any(t not in token_to_id for t in SPECIAL_TOKENS):
        raise ValueError("词表缺少特殊 token")
    ids = [token_to_id["<bos>"]] + encode(prompt, token_to_id)
    new_ids = []
    ended = False
    device = next(model.parameters()).device
    rng = torch.Generator(device="cpu").manual_seed(seed)
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            for _ in range(max_new_tokens):
                inputs = torch.tensor([ids], dtype=torch.long, device=device)
                last_logits = model(inputs)[0, -1, :]
                if method == "greedy":
                    next_id = int(last_logits.argmax().item())
                else:
                    probabilities = torch.softmax(last_logits / temperature, dim=-1).cpu()
                    next_id = int(torch.multinomial(probabilities, 1, generator=rng).item())
                ids.append(next_id)
                new_ids.append(next_id)
                if next_id == token_to_id["<eos>"]:
                    ended = True
                    break
    finally:
        model.train(was_training)
    visible_ids = new_ids[:-1] if ended else new_ids
    return {
        "text": prompt + decode(visible_ids, id_to_token),
        "new_token_ids": new_ids,
        "all_token_ids": ids,
        "stopped_on_eos": ended,
        "method": method,
    }

In [ ]:
for prompt in ["", "我爱", "你爱", "我爱学"]:
    result = generate(model, prompt, token_to_id, id_to_token)
    print(f"贪心 {prompt!r} -> {result['text']!r}，遇到EOS：{result['stopped_on_eos']}")
print("按概率采样：")
for sampling_seed in [1, 2, 3]:
    result = generate(model, "", token_to_id, id_to_token, method="sample", seed=sampling_seed)
    print(sampling_seed, result["text"], result["new_token_ids"])

### 温度改变抽样分布，不会让模型凭空学会新知识

固定一组三分类 logits，只改变温度：较低温度会让分布更集中，较高温度让分布更平缓。
本课贪心直接比较 logits，因此不会受到正温度缩放的影响。

未训练好的模型也可能生成 `<unk>` 或 `<bos>`。代码没有偷偷过滤这些类别；它们会原样显示。
不认识的提示字符使用 `<unk>` 编码，而本训练语料没有未知字符作为输入，不能指望这行向量有可靠的预测能力。

In [ ]:
fixed_logits = torch.tensor([2., 1., 0.])
for temperature in [0.5, 1.0, 2.0]:
    print("温度", temperature, "概率", torch.softmax(fixed_logits / temperature, dim=-1))

## 12. 用实验证明“输入整句话”不等于“利用整句话”

取两个不同前缀：“我爱”与“你爱”。末尾都是“爱”，所以本模型最后位置的 logits 完全相同。

这是结构决定的：`Embedding` 查同一行，`Linear` 用同一组参数；从更早位置到这个输出没有计算路径。
把 embedding 维度放大，或在每个位置增加 MLP，依旧不会自动建立跨位置路径。

**变化实验：** 后续可以只保留“我爱学习。”和“你爱模型。”两句。
此时要在“爱”后选“学”还是“模”，就需要看更早的“我/你”；当前模型依然无法区分。
这才适合在后续注意力模型中检验更长上下文的价值。

In [ ]:
first_prefix = torch.tensor([[token_to_id["<bos>"]] + encode("我爱", token_to_id)], dtype=torch.long, device=device)
second_prefix = torch.tensor([[token_to_id["<bos>"]] + encode("你爱", token_to_id)], dtype=torch.long, device=device)
with torch.no_grad():
    first_last = model(first_prefix)[:, -1, :]
    second_last = model(second_prefix)[:, -1, :]
print("相同末字得到相同预测分数：", torch.allclose(first_last, second_last))
print("最大绝对差：", (first_last - second_last).abs().max().item())

## 13. 保存的不只是权重，还有编号约定

如果权重里第 3 行对应“我”，加载时却把第 3 个编号改成“你”，即使形状完全正确，含义也错了。
所以本课保存：**`state_dict`、结构配置和有顺序的词表**，然后重建同结构模型、恢复参数、比较 logits。[5]

这里是推理用检查点，不保存 Adam 的状态或数据读取位置，不能声称能精确续接原训练轨迹。
只加载自己生成或信任的文件；`weights_only=True` 不等于对任何恶意文件都完全安全。

In [ ]:
def save_checkpoint(model: TinyNextTokenLM, id_to_token: list[str], path: str | Path) -> Path:
    """权重、结构配置与词表顺序一起保存；只加载自己信任的文件。"""
    if len(id_to_token) != model.vocab_size or id_to_token[:3] != SPECIAL_TOKENS:
        raise ValueError("词表与模型不一致")
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({
        "state_dict": {name: value.detach().cpu() for name, value in model.state_dict().items()},
        "config": {"vocab_size": model.vocab_size, "embedding_dim": model.embedding_dim},
        "id_to_token": list(id_to_token),
    }, path)
    return path


def load_checkpoint(path: str | Path, device: str | torch.device = "cpu") -> tuple[TinyNextTokenLM, dict[str, int], list[str]]:
    """演示重建模型并恢复编号约定；不是不可信文件的安全验证器。"""
    checkpoint = torch.load(Path(path), map_location="cpu", weights_only=True)
    vocab = checkpoint["id_to_token"]
    config = checkpoint["config"]
    if not isinstance(vocab, list) or any(not isinstance(t, str) for t in vocab):
        raise ValueError("检查点词表格式错误")
    if len(vocab) != config["vocab_size"] or len(set(vocab)) != len(vocab) or vocab[:3] != SPECIAL_TOKENS:
        raise ValueError("检查点的词表顺序或结构不匹配")
    model = TinyNextTokenLM(**config)
    model.load_state_dict(checkpoint["state_dict"])
    model.to(device).eval()
    return model, {token: index for index, token in enumerate(vocab)}, vocab

notebook_output_dir = Path("lesson_13_outputs") / "notebook"
checkpoint_path = save_checkpoint(model, id_to_token, notebook_output_dir / "tiny_char_lm.pt")
restored_model, restored_map, restored_vocab = load_checkpoint(checkpoint_path, device)
with torch.no_grad():
    torch.testing.assert_close(model(x), restored_model(x))
print("保存与重载后 logits 一致。")
print("词表顺序一致：", restored_vocab == id_to_token)
print("加载后生成：", generate(restored_model, "我爱", restored_map, restored_vocab)["text"])
print("本次检查点：", checkpoint_path.resolve())

---
## 14. 独立练习：请先停在这里，自己编写

练习 A、B、C 都给出函数签名，不自动替你完成。
末尾提供参考答案，使用不同名字，不覆盖你的函数。

**A：对齐标签。** 实现 `shift_targets_ex(full_ids)`，输入为 `[B,L]` 的 long 张量，且 `B≥1,L≥2`，返回 `[B,L-1]` 的 x、y。对无效类型或过短输入抛出 `ValueError`。

**B：计算 token 平均交叉熵。** 实现 `token_loss_ex(logits, targets)`，对合法的 `[B,T,V]` / `[B,T]` 输入直接算损失，不再次移位、不先 softmax。必须保留可反向传播的标量张量，不能 `.item()` 后再返回。

**C：下一 token 推理。** 实现 `predict_next_ex(model, prefix_ids)`，返回每条前缀下一个 token 的贪心编号，形状 `[B]`。只读取最后位置、关闭梯度、不修改参数，并恢复调用前的 train/eval 模式。

做完一项，将对应单元的 `READY_*` 改为 `True`，运行自动检查。

In [ ]:
def check_shift_exercise(function):
    full = torch.tensor([[1, 3, 4, 2], [1, 10, 4, 2]], dtype=torch.long)
    original = full.clone()
    a, b = function(full)
    torch.testing.assert_close(a, torch.tensor([[1, 3, 4], [1, 10, 4]]))
    torch.testing.assert_close(b, torch.tensor([[3, 4, 2], [10, 4, 2]]))
    torch.testing.assert_close(full, original)
    short_a, short_b = function(torch.tensor([[1, 2]]))
    torch.testing.assert_close(short_a, torch.tensor([[1]]))
    torch.testing.assert_close(short_b, torch.tensor([[2]]))
    for invalid in [full.float(), full[0], torch.tensor([[1]]), torch.empty(0, 3, dtype=torch.long)]:
        try:
            function(invalid)
        except ValueError:
            continue
        raise AssertionError("练习 A 没有拒绝不符合约定的输入")
    print("练习 A 检查通过。")


def check_loss_exercise(function):
    logits = torch.tensor([[[2., 0., -1.], [0., 1., 3.]], [[0., 0., 0.], [1., -1., 0.]]], requires_grad=True)
    labels = torch.tensor([[0, 2], [1, 0]], dtype=torch.long)
    loss = function(logits, labels)
    if not isinstance(loss, torch.Tensor) or loss.ndim != 0:
        raise AssertionError("应返回标量 Tensor，而不是 Python 数值")
    expected = -(F.log_softmax(logits, dim=-1)[0, 0, 0] + F.log_softmax(logits, dim=-1)[0, 1, 2]
                 + F.log_softmax(logits, dim=-1)[1, 0, 1] + F.log_softmax(logits, dim=-1)[1, 1, 0]) / 4
    torch.testing.assert_close(loss, expected)
    torch.testing.assert_close(function(torch.zeros_like(logits), labels), torch.tensor(math.log(3)))
    loss.backward()
    if logits.grad is None or not torch.isfinite(logits.grad).all():
        raise AssertionError("损失没有保留正确的梯度路径")
    print("练习 B 检查通过。")


def check_prediction_exercise(function):
    # 每个位置故意有不同的最大类别，避免“误取第一个位置”也碰巧通过。
    class FixedOutput(nn.Module):
        def __init__(self):
            super().__init__()
            self.scores = nn.Parameter(torch.tensor([
                [[9., 0., 0., 0.], [0., 0., 0., 9.]],
                [[0., 0., 9., 0.], [0., 9., 0., 0.]],
            ]))
            self.grad_enabled_during_call = None
        def forward(self, inputs):
            self.grad_enabled_during_call = torch.is_grad_enabled()
            return self.scores

    fixture = FixedOutput().train()
    original = fixture.scores.detach().clone()
    result = function(fixture, torch.tensor([[1, 2], [2, 1]], dtype=torch.long))
    torch.testing.assert_close(result, torch.tensor([3, 1]))
    torch.testing.assert_close(fixture.scores, original)
    if result.dtype != torch.long or not fixture.training or fixture.grad_enabled_during_call:
        raise AssertionError("检查结果类型、原模式恢复和 no_grad 的使用")
    fixture.eval()
    function(fixture, torch.tensor([[1, 2], [2, 1]]))
    if fixture.training:
        raise AssertionError("调用前是 eval 时也应保持 eval")
    print("练习 C 检查通过。")

In [ ]:
def shift_targets_ex(full_ids):
    # TODO：检查张量类型、dtype、形状；使用两个切片构造 x 与 y。
    raise NotImplementedError("请自己完成练习 A")

READY_A = False
if READY_A:
    check_shift_exercise(shift_targets_ex)
else:
    print("练习 A 尚未验收：完成后将 READY_A 改为 True。")

In [ ]:
def token_loss_ex(logits, targets):
    # TODO：展平 B、T 两条轴；直接用 logits 计算交叉熵。
    raise NotImplementedError("请自己完成练习 B")

READY_B = False
if READY_B:
    check_loss_exercise(token_loss_ex)
else:
    print("练习 B 尚未验收：完成后将 READY_B 改为 True。")

In [ ]:
def predict_next_ex(model, prefix_ids):
    # TODO：保存模式、eval、no_grad、只取最后位置的 logits、argmax、恢复模式。
    raise NotImplementedError("请自己完成练习 C")

READY_C = False
if READY_C:
    check_prediction_exercise(predict_next_ex)
else:
    print("练习 C 尚未验收：完成后将 READY_C 改为 True。")

### 可选变化实验：更大的 MLP 是否会自动获得上下文？

把主模型输出层改成 `Linear(D,32) → GELU → Linear(32,V)`，保持 Embedding 不变。
重新训练，再比较“我爱”和“你爱”最后位置的 logits。

预测：两者依旧相同，因为增加的计算仍然逐位置进行。它可能改变拟合过程，但不会增加可访问的上下文。
这道题检验你是否真正区分“参数更多”和“信息流改变”。

---
## 15. 参考答案与检查

只有在自己写过后再阅读。答案采用 `_answer` 后缀，不替换你的 `_ex` 函数。
`try/finally` 用于确保即使前向计算报错也能恢复模式；正常路径下它也可以理解为“使用前记住状态，使用后还原”。

In [ ]:
def shift_targets_answer(full_ids):
    if not isinstance(full_ids, torch.Tensor) or full_ids.ndim != 2:
        raise ValueError("需要二维编号张量")
    if full_ids.dtype != torch.long or full_ids.shape[0] < 1 or full_ids.shape[1] < 2:
        raise ValueError("需要非空批量、至少两个 token，且类型为 long")
    return full_ids[:, :-1].contiguous(), full_ids[:, 1:].contiguous()

check_shift_exercise(shift_targets_answer)

In [ ]:
def token_loss_answer(logits, targets):
    vocab_size = logits.shape[-1]
    return F.cross_entropy(logits.reshape(-1, vocab_size), targets.reshape(-1))

check_loss_exercise(token_loss_answer)

In [ ]:
def predict_next_answer(model, prefix_ids):
    was_training = model.training
    model.eval()
    try:
        with torch.no_grad():
            last_logits = model(prefix_ids)[:, -1, :]
            return last_logits.argmax(dim=-1)
    finally:
        model.train(was_training)

check_prediction_exercise(predict_next_answer)

## 16. 固定检查与脚本入口

下面的检查代码与配套脚本一致，可以留到第二遍阅读。
它验证编号、边界对齐、Embedding 查表和梯度、交叉熵数值、训练、模型的上下文限制、生成停止条件、保存重载等行为。
**这是代码行为测试，不是语言能力评测。**

配套脚本运行命令（在它所在目录）：

```bash
python lesson_13_embeddings_and_cross_entropy.py
python lesson_13_embeddings_and_cross_entropy.py --device auto
python lesson_13_embeddings_and_cross_entropy.py --checks-only
```

脚本在当前工作目录下生成 `lesson_13_outputs/tiny_char_lm.pt`、`metrics.json`、`samples.txt`，
会覆盖同名生成文件。可以用 `--output-dir` 选择另一个目录。
导入脚本不会自动训练、保存或运行检查。

下面不需要通过 import 找到脚本文件：所有定义都已包含在 Notebook 中。

In [ ]:
def run_checks(device: str | torch.device = "cpu") -> list[str]:
    """固定行为检查。显式抛出 AssertionError，因此 python -O 不会删掉检查。"""
    device = torch.device(device)
    passed = []

    def check(condition: bool, name: str) -> None:
        if not bool(condition):
            raise AssertionError(name)
        passed.append(name)

    def rejects(function, name: str) -> None:
        try:
            function()
        except ValueError:
            passed.append(name)
        else:
            raise AssertionError("未拒绝错误输入：" + name)

    def close(a, b) -> bool:
        return bool(torch.allclose(a, b, atol=1e-6, rtol=1e-5))

    token_to_id, vocab = build_vocab(TRAIN_TEXTS)
    v = len(vocab)
    check(v == 11, "词表大小为 11")
    check(vocab[:3] == SPECIAL_TOKENS, "特殊编号顺序固定")
    check(build_vocab(TRAIN_TEXTS) == (token_to_id, vocab), "重建词表可复现")
    check(decode(encode("我爱学习。", token_to_id), vocab) == "我爱学习。", "已知字符往返")
    check(encode("火", token_to_id) == [0], "未知字符映射")
    check(encode("", token_to_id) == [], "空字符串编码")
    check(decode([1, 2], vocab) == "<bos><eos>", "特殊 token 可见")
    rejects(lambda: build_vocab([]), "空语料")
    rejects(lambda: build_vocab([1]), "语料元素类型")
    rejects(lambda: encode(True, token_to_id), "非字符串编码")
    rejects(lambda: decode([True], vocab), "布尔编号")
    rejects(lambda: decode([-1], vocab), "负编号")
    rejects(lambda: decode([v], vocab), "越界编号")

    x, y = make_lm_batch(TRAIN_TEXTS, token_to_id, device)
    check(tuple(x.shape) == (4, 6) and x.shape == y.shape, "批量形状")
    check(x.dtype == y.dtype == torch.long, "输入标签都是 long")
    check(x[0].tolist() == [1, 3, 4, 5, 6, 7], "第一条输入精确对齐")
    check(y[0].tolist() == [3, 4, 5, 6, 7, 2], "第一条标签精确对齐")
    check(torch.equal(x[:, 1:], y[:, :-1]), "只移位一次")
    check(bool((x[:, 0] == 1).all()) and bool((y[:, -1] == 2).all()), "BOS/EOS 边界")
    empty_x, empty_y = make_lm_batch([""], token_to_id, device)
    check(empty_x.tolist() == [[1]] and empty_y.tolist() == [[2]], "空文本仍有 BOS 到 EOS")
    rejects(lambda: make_lm_batch(["我", "我爱"], token_to_id), "等长约束")

    table = nn.Embedding(5, 3).to(device)
    with torch.no_grad():
        table.weight.copy_(torch.arange(15, dtype=torch.float32, device=device).reshape(5, 3))
    ids = torch.tensor([[1, 1, 3]], dtype=torch.long, device=device)
    embedded = table(ids)
    check(tuple(embedded.shape) == (1, 3, 3), "Embedding 形状")
    check(torch.equal(embedded, table.weight[ids]), "Embedding 等于按编号查行")
    check(torch.equal(embedded[0, 0], embedded[0, 1]), "同编号同初始向量")
    one_hot = F.one_hot(ids, num_classes=5).to(torch.float32)
    check(torch.equal(embedded, one_hot @ table.weight), "one-hot 乘表的等价性")
    embedded.sum().backward()
    check(torch.equal(table.weight.grad[1], torch.full((3,), 2., device=device)), "重复查行梯度累加")
    check(torch.equal(table.weight.grad[0], torch.zeros(3, device=device)), "本次未查行梯度为零")
    check(not ids.requires_grad, "整数 ID 不作为可微参数")

    z = torch.tensor([[[2., 0., -1.], [0., 1., 3.]]], device=device, requires_grad=True)
    target = torch.tensor([[0, 2]], dtype=torch.long, device=device)
    loss = sequence_cross_entropy(z, target)
    manual = -(F.log_softmax(z, dim=-1)[0, 0, 0] + F.log_softmax(z, dim=-1)[0, 1, 2]) / 2
    check(close(loss, manual), "交叉熵等于目标 log 概率负平均")
    check(close(loss, F.cross_entropy(z.transpose(1, 2), target)), "展平与换轴写法等价")
    check(close(loss, sequence_cross_entropy(z + 1000., target)), "平移 logits 不改损失")
    check(close(sequence_cross_entropy(torch.zeros_like(z), target), torch.tensor(math.log(3), device=device)), "均匀三分类损失 ln(3)")
    huge = torch.tensor([[[1000., 0., -1000.]]], device=device)
    check(close(sequence_cross_entropy(huge, torch.tensor([[2]], device=device)), torch.tensor(2000., device=device)), "极端 logits 仍有有限损失")
    improved = z.detach().clone()
    improved[0, 0, 0] += 1
    improved[0, 1, 2] += 1
    check(sequence_cross_entropy(improved, target).item() < loss.item(), "提高正确类分数降低损失")
    loss.backward()
    check(z.grad is not None and bool(torch.isfinite(z.grad).all()), "CE 反向梯度有限")
    check(close(z.grad.sum(dim=-1), torch.zeros_like(target, dtype=torch.float32)), "每个位置的 logits 梯度和为零")
    rejects(lambda: sequence_cross_entropy(z, target.float()), "浮点类别标签")
    rejects(lambda: sequence_cross_entropy(z, torch.tensor([[0, 3]], device=device)), "标签越界")
    rejects(lambda: sequence_cross_entropy(z, target[:, :1]), "损失形状不匹配")
    rejects(lambda: sequence_cross_entropy(z.long(), target), "整数 logits")
    rejects(lambda: sequence_cross_entropy(torch.empty(1, 0, 3, device=device), torch.empty(1, 0, dtype=torch.long, device=device)), "空序列损失")
    rejects(lambda: sequence_cross_entropy(z.detach() * float('nan'), target), "非有限 logits")

    torch.manual_seed(13)
    model = TinyNextTokenLM(v, 16).to(device)
    output = model(x)
    check(tuple(output.shape) == (4, 6, v), "模型输出词表分数")
    check(sum(p.numel() for p in model.parameters()) == 363, "模型参数量")
    rejects(lambda: model(x.float()), "模型拒绝浮点 ID")
    rejects(lambda: model(x[0]), "模型拒绝一维输入")
    rejects(lambda: model(torch.tensor([[v]], device=device)), "模型拒绝越界 ID")
    rejects(lambda: TinyNextTokenLM(0), "模型结构参数检查")
    sequence_cross_entropy(output, y).backward()
    check(all(p.grad is not None and bool(torch.isfinite(p.grad).all()) for p in model.parameters()), "所有注册参数接收有限梯度")
    check(model.embedding.weight.grad.abs().sum().item() > 0, "Embedding 梯度非零")
    first = torch.tensor([[token_to_id['我'], token_to_id['爱']]], device=device)
    second = torch.tensor([[token_to_id['你'], token_to_id['爱']]], device=device)
    check(close(model(first)[:, -1], model(second)[:, -1]), "同末字不使用早期上下文")
    changed = x.clone()
    changed[:, -1] = token_to_id["你"]
    check(close(model(x)[:, :-1], model(changed)[:, :-1]), "逐位置模型无跨位置混合")

    baseline = bigram_frequency_baseline(x, y, v)
    check(abs(baseline['loss_floor'] - math.log(2) / 3) < 1e-10, "频率损失下界")
    check(abs(baseline['accuracy_ceiling'] - 5 / 6) < 1e-10, "一字符准确率上界")
    before = model.embedding.weight.detach().clone()
    report = train_model(model, x, y, steps=120)
    check(report['after']['loss'] < report['before']['loss'] * 0.3, "固定教学训练损失下降")
    check(abs(report['after']['loss'] - baseline['loss_floor']) < 0.02, "拟合接近频率下界")
    check(report['after']['token_accuracy'] <= baseline['accuracy_ceiling'] + 1e-6, "不超过一字符条件准确率上界")
    check(not torch.equal(before, model.embedding.weight.detach()), "Embedding 权重确实更新")
    rejects(lambda: train_model(model, x, y, steps=0), "无效训练步数")
    rejects(lambda: train_model(model, x, y, lr=float('nan')), "无效学习率")

    result = generate(model, "我爱", token_to_id, vocab, max_new_tokens=0)
    check(result['text'] == '我爱' and result['new_token_ids'] == [], "零生成预算")
    fixed = TinyNextTokenLM(v, 4).to(device)
    with torch.no_grad():
        fixed.lm_head.weight.zero_()
        fixed.lm_head.bias.fill_(-100.)
        fixed.lm_head.bias[token_to_id['<eos>']] = 100.
    result = generate(fixed, '我', token_to_id, vocab)
    check(result['stopped_on_eos'] and result['new_token_ids'] == [2], "EOS 停止")
    check(result['text'] == '我', "EOS 不混入显示文本")
    with torch.no_grad():
        fixed.lm_head.bias.fill_(-100.)
        fixed.lm_head.bias[token_to_id['我']] = 100.
    result = generate(fixed, '', token_to_id, vocab, max_new_tokens=3)
    check(result['text'] == '我我我' and not result['stopped_on_eos'], "最大生成长度保护")
    weights_before = {name: value.clone() for name, value in model.state_dict().items()}
    a = generate(model, '', token_to_id, vocab, method='sample', seed=17)
    b = generate(model, '', token_to_id, vocab, method='sample', seed=17)
    check(a == b, "同模型同采样种子可复现")
    check(all(torch.equal(value, model.state_dict()[name]) for name, value in weights_before.items()), "生成不修改模型参数")
    check(model.training, "生成恢复原训练模式")
    rejects(lambda: generate(model, '', token_to_id, vocab, temperature=0), "无效温度")
    rejects(lambda: generate(model, '', token_to_id, vocab, max_new_tokens=-1), "负生成预算")
    rejects(lambda: generate(model, '', token_to_id, vocab, method='bad'), "无效生成方法")

    with tempfile.TemporaryDirectory(prefix='lesson13_check_') as temp_dir:
        path = save_checkpoint(model, vocab, Path(temp_dir) / 'test.pt')
        restored, restored_map, restored_vocab = load_checkpoint(path, device)
        check(restored_map == token_to_id and restored_vocab == vocab, "保存重载保留词表顺序")
        with torch.no_grad():
            check(close(model(x), restored(x)), "保存重载 logits 一致")
        check(generate(model, '我爱', token_to_id, vocab) == generate(restored, '我爱', restored_map, restored_vocab), "保存重载生成一致")
    return passed

passed_checks = run_checks(device)
print(f"通过固定检查：{len(passed_checks)} 项")

## 17. 常见错误与掌握标准

| 现象 | 先检查 |
|---|---|
| Embedding 报编号类型/越界错误 | ID 是否为 long？是否在 `[0,V)` 内？ |
| CE 维度错误 | 是 `[B*T,V]` 配 `[B*T]`，还是 `[B,V,T]` 配 `[B,T]`？ |
| 模型总在预测当前字 | 标签是否没移位？ |
| 总在预测隔一个字 | 标签是否被移位两次？ |
| loss 数字不同却不报错 | 是否错误地先 softmax 再传给交叉熵？是否误用求和而非平均？ |
| 生成第一步就崩溃 | 是否给空张量？空提示也应先加入 BOS。 |
| 改词表后重载生成乱码 | 编号顺序是否与训练时一致？ |
| 同末字的不同句子预测一样 | 对本课结构是预期行为，不是训练循环故障。 |

不看答案回答以下问题：

1. `Embedding(11,16)` 的参数形状是什么？输入 `[4,6]` 时输出是什么？
2. token 编号和 Embedding 参数哪个参与梯度更新？
3. 为什么 `x` 的最后位置是句号，而 `y` 的最后位置是 EOS？
4. 为什么 CE 直接接收 logits，不先 softmax？
5. `model(x).shape=[4,6,11]` 中，哪一维是候选 token？
6. 当前模型接收 100 个 token，就利用了 100 个 token 的上下文吗？
7. 为什么 83.33% 是本数据上可解释的训练准确率，而不是“模型没学会”的充分证据？
8. 有了推理检查点，是否能逐步精确续接 Adam 训练？还缺什么？

<details>
<summary>参考要点（完成后展开）</summary>

1. 参数 `[11,16]`；输出 `[4,6,16]`。2. 浮点权重参与更新，ID 是索引。
3. 每个输入位置预测紧随其后的 token。4. 接口负责稳定的 log-softmax 相关计算，先 softmax 会改变目标。
5. 最后一维 11。6. 不能，本模型每个位置只看自己的输入字符。
7. 同一输入可能有多个目标，必须结合数据频率与模型信息路径分析；训练分数也不是泛化分数。
8. 不行；还需要优化器状态、步数、数据迭代位置以及相关随机状态等。

</details>

下一课：**第 14 课——数据集、批量训练与验证集**。
把“固定四句话全部一起训练”升级为规范的数据读取与评估流程，再处理变长文本和 padding。

## 官方参考与本课范围

文中的 `[1]` 到 `[6]` 对应下列官方资料；公式推导、四句语料和诊断实验是本课自行构造。

[1] [PyTorch `nn.Embedding`：查表、参数形状与输入输出](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)  
[2] [PyTorch `nn.CrossEntropyLoss`：logits、类别编号与形状](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html)  
[3] [PyTorch `F.log_softmax`：数值稳定性](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.log_softmax.html)  
[4] [PyTorch `torch.multinomial`：按权重抽取编号](https://docs.pytorch.org/docs/stable/generated/torch.multinomial.html)  
[5] [PyTorch 保存与加载模型](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html)  
[6] [PyTorch `nn.Linear`：最后一维上的仿射变换](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

补充：[官方词向量与 N-gram 语言建模入门](https://docs.pytorch.org/tutorials/beginner/nlp/word_embeddings_tutorial.html)；[PyTorch 安装选择器](https://pytorch.org/get-started/locally/)。

**验证边界：** 随文件保存的输出来自 CPU 实测。脚本提供 CUDA/auto 选项，但本次执行环境没有可用 CUDA，未在 RTX 3060 上实测。
Notebook 使用固定教学数据，不下载或连接外部服务；安装依赖与访问参考链接例外。